In [ ]:
%pip install -q sagemaker boto3 pandas numpy matplotlib scikit-learn

# Week 2 Thursday -- Advanced SageMaker Built-in Algorithms

Monday you trained a Random Forest via Script Mode -- you wrote your own `train.py` and SageMaker ran it. Today you use SageMaker's **built-in algorithms**: managed, optimized implementations where you supply data and hyperparameters only. No training script. XGBoost typically outperforms Random Forest on tabular data. You also explore unsupervised algorithms (K-Means, Random Cut Forest) and then use Hyperparameter Optimization (HPO) to systematically tune XGBoost.

**Pairs with:** Wednesday's experiment tracking (HPO is systematic, automated experimentation) and Monday's RF model (XGBoost is a stronger tabular model for the same fraud data).

By the end of this notebook you will have:

1. **Trained XGBoost** as a built-in algorithm on the same fraud data and compared it to Monday's RF.
2. **Clustered** fraud data with K-Means and **detected anomalies** with Random Cut Forest.
3. **Launched an HPO job** to find optimal XGBoost hyperparameters automatically.
4. **Connected HPO** to Wednesday's experiment tracking concepts.

| Block | Content | Minutes |
|-------|---------|--------|
| Stage 1 | XGBoost for Fraud Detection | 55 |
| Stage 2 | Other Built-in Algorithms | 55 |
| Stage 3 | Hyperparameter Optimization | 55 |

## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region.

In [ ]:
import boto3
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

sm_client = boto3.client("sagemaker")
s3 = boto3.client("s3")

print(f"Region:  {region}")
print(f"Bucket:  s3://{bucket}")
print(f"Role:    ...{role[-30:]}")

---
# STAGE 1 -- XGBoost for Fraud Detection

**Connecting to Monday:** On Monday you trained a Random Forest using Script Mode -- you wrote `train.py`, packaged your own code, and SageMaker ran it. Today we use a **built-in algorithm** instead. SageMaker's built-in XGBoost is a managed, optimized implementation of gradient boosted trees. You supply data and hyperparameters. SageMaker handles the rest.

**Why XGBoost often outperforms Random Forest:**
- Random Forest trains trees independently on bootstrapped samples. Each tree votes, and the majority wins.
- XGBoost trains trees sequentially. Each new tree focuses on the errors of the ensemble so far. This sequential correction allows XGBoost to learn complex patterns that independent trees miss.
- XGBoost also includes built-in regularization (L1/L2) to prevent overfitting.

**Built-in vs Script Mode:**

| Aspect | Script Mode (Monday's RF) | Built-in (Today's XGBoost) |
|--------|---------------------------|---------------------------|
| Training code | You write `train.py` | None -- managed by container |
| Model framework | Any (sklearn, PyTorch, etc.) | XGBoost only |
| Data format | Flexible (you parse it) | Target first, no header |
| Hyperparameters | Argparse in your script | `.set_hyperparameters()` |
| Optimization | Your responsibility | AWS-optimized, distributed |

## STEP 1 -- Prepare Fraud Data for XGBoost

> **What is happening:** We load the fraud data from Monday (or regenerate it if missing) and reformat it for the XGBoost built-in algorithm. The required format is CSV with the target column first, no header row, no index column. This is a strict contract with the built-in container -- violating it produces silent errors or garbage predictions.

In [ ]:
import numpy as np
import pandas as pd
import os

s3 = boto3.client("s3")

train_exists = False
val_exists = False
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/train/train.csv")
    train_exists = True
except Exception:
    pass
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/validation/validation.csv")
    val_exists = True
except Exception:
    pass

if train_exists and val_exists:
    s3.download_file(bucket, f"{prefix}/data/train/train.csv", "train_original.csv")
    s3.download_file(bucket, f"{prefix}/data/validation/validation.csv", "validation_original.csv")
    train_df = pd.read_csv("train_original.csv")
    val_df = pd.read_csv("validation_original.csv")
    print(f"Downloaded training data:   {train_df.shape}")
    print(f"Downloaded validation data: {val_df.shape}")
else:
    print("Monday's data not found in S3. Regenerating synthetic fraud data...")
    np.random.seed(42)
    n = 2000
    data = pd.DataFrame({
        "amount": np.random.exponential(500, n).round(2),
        "hour": np.random.randint(0, 24, n),
        "distance_from_home": np.random.exponential(50, n).round(2),
        "transaction_count_24h": np.random.poisson(5, n),
        "is_international": np.random.binomial(1, 0.1, n),
        "merchant_risk_score": np.random.uniform(0, 1, n).round(3),
    })
    data["target"] = (
        (data["amount"] > 800) & (data["hour"] < 6) | (data["merchant_risk_score"] > 0.85)
    ).astype(int)
    noise = np.random.random(n) < 0.08
    data["target"] = (data["target"] ^ noise.astype(int))
    train_df = data.iloc[:1600].copy()
    val_df = data.iloc[1600:].copy()
    print(f"Generated training data:   {train_df.shape}")
    print(f"Generated validation data: {val_df.shape}")

feature_cols = [c for c in train_df.columns if c != "target"]
print(f"\nFeatures: {feature_cols}")
print(f"Fraud rate (train): {train_df['target'].mean():.3f}")
print(f"Fraud rate (val):   {val_df['target'].mean():.3f}")

> **What is happening:** We reorder columns so the target comes first (XGBoost built-in requirement) and save without headers or index. This is the data format contract.

In [ ]:
xgb_cols = ["target"] + feature_cols
train_xgb = train_df[xgb_cols]
val_xgb = val_df[xgb_cols]

train_xgb.to_csv("train_xgb.csv", header=False, index=False)
val_xgb.to_csv("val_xgb.csv", header=False, index=False)

print("XGBoost CSV format (first 3 rows of training data):")
print(open("train_xgb.csv").readlines()[:3])
print(f"\nColumn order: {xgb_cols}")
print("Target is column 0. No header. No index.")

## STEP 2 -- Upload to S3 with Train/Validation Channels

> **What is happening:** SageMaker built-in algorithms expect data in S3, organized by channel. The `train` channel contains training data; the `validation` channel is used for evaluation during training. We upload the reformatted CSVs to separate S3 prefixes and create `TrainingInput` objects that tell the estimator where to find each channel.

In [ ]:
from sagemaker.inputs import TrainingInput

train_s3_key = f"{prefix}/xgboost/train/train.csv"
val_s3_key = f"{prefix}/xgboost/validation/validation.csv"

s3.upload_file("train_xgb.csv", bucket, train_s3_key)
s3.upload_file("val_xgb.csv", bucket, val_s3_key)

train_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/xgboost/train/",
    content_type="text/csv",
)
val_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/xgboost/validation/",
    content_type="text/csv",
)

print(f"Train channel: s3://{bucket}/{train_s3_key}")
print(f"Val channel:   s3://{bucket}/{val_s3_key}")

## STEP 3 -- Configure XGBoost Estimator

> **What is happening:** We retrieve the XGBoost built-in container image URI and create an `Estimator`. Unlike Script Mode, there is no `entry_point` or `source_dir` -- the container already contains the training code. We configure behavior entirely through hyperparameters.

**Hyperparameters explained:**
- `max_depth=5` -- Maximum tree depth. Deeper trees capture more complex patterns but risk overfitting.
- `eta=0.2` -- Learning rate. Each tree's contribution is scaled by this factor. Lower values need more rounds.
- `num_round=100` -- Number of boosting rounds (trees to add sequentially).
- `objective="binary:logistic"` -- Binary classification with sigmoid output (probability).
- `eval_metric="auc"` -- Area under the ROC curve. Better than accuracy for imbalanced fraud data.

In [ ]:
from sagemaker import image_uris
from sagemaker.estimator import Estimator

xgb_image = image_uris.retrieve("xgboost", region, version="1.5-1")
print(f"XGBoost container image: {xgb_image}")

xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/xgboost/output",
    sagemaker_session=session,
)

xgb_estimator.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    num_round=100,
    objective="binary:logistic",
    eval_metric="auc",
    subsample=0.8,
    colsample_bytree=0.8,
)

print("\nXGBoost estimator configured.")
print(f"  Instance type: ml.m5.xlarge")
print(f"  Hyperparameters: {xgb_estimator.hyperparameters()}")

## STEP 4 -- Launch Training Job

> **What is happening:** We call `.fit()` with the train and validation channels. SageMaker provisions the instance, pulls the XGBoost container, downloads data from S3, trains the model, evaluates on the validation channel, and uploads the model artifact to S3. This takes approximately 3-5 minutes.

In [ ]:
xgb_estimator.fit(
    {"train": train_input, "validation": val_input},
    logs=True,
)

print(f"\nTraining job: {xgb_estimator.latest_training_job.name}")
print(f"Model artifact: {xgb_estimator.model_data}")

## Comparing Approaches: Random Forest (Monday) vs XGBoost (Today)

| Dimension | Monday's Random Forest | Today's XGBoost |
|-----------|----------------------|----------------|
| **Mode** | Script Mode -- wrote `train.py` | Built-in -- no training script |
| **Algorithm** | Ensemble of independent trees | Sequential boosted trees |
| **Data format** | Standard CSV with header | Target first, no header |
| **How it learns** | Each tree votes independently | Each tree corrects prior errors |
| **Regularization** | None built in | L1/L2 + subsample + colsample |
| **Typical performance** | Good baseline | Often stronger on tabular data |

> **Discussion:** Both models solve the same problem (fraud detection on tabular data). The difference is in how they learn. Random Forest is a strong, simple baseline. XGBoost's sequential correction mechanism makes it the dominant algorithm for tabular ML competitions and production systems.

## STEP 5 -- Deploy and Evaluate XGBoost

> **What is happening:** We deploy the trained XGBoost model to a real-time endpoint, send the validation data through it, and compute the same metrics we used on Monday: precision, recall, F1, and a confusion matrix. This gives us a fair comparison.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

xgb_predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

print(f"Endpoint name: {xgb_predictor.endpoint_name}")
print("Endpoint is InService.")

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
import matplotlib.pyplot as plt

X_val = val_df[feature_cols].values
y_val = val_df["target"].values

predictions_raw = xgb_predictor.predict(X_val)
probabilities = np.array([float(row[0]) for row in predictions_raw])
predictions = (probabilities >= 0.5).astype(int)

print("=== XGBoost Evaluation on Validation Data ===")
print(f"  Precision: {precision_score(y_val, predictions):.4f}")
print(f"  Recall:    {recall_score(y_val, predictions):.4f}")
print(f"  F1:        {f1_score(y_val, predictions):.4f}")
print(f"  AUC:       {roc_auc_score(y_val, probabilities):.4f}")
print()
print(classification_report(y_val, predictions, target_names=["Non-Fraud", "Fraud"]))

cm = confusion_matrix(y_val, predictions)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Non-Fraud", "Fraud"])
ax.set_yticklabels(["Non-Fraud", "Fraud"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("XGBoost Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=16)
plt.colorbar(im)
plt.tight_layout()
plt.show()

> **Discussion:** Compare these metrics to Monday's Random Forest. Which model has better recall? Better precision? For FraudShield, recall matters most -- a missed fraud is more costly than a false alarm. If XGBoost improved recall without sacrificing precision, it is the stronger model for this problem.

## STEP 6 -- Cleanup XGBoost Endpoint

> **What is happening:** We delete the XGBoost endpoint immediately to avoid ongoing charges. We will create new endpoints in Stage 2 and Stage 3.

In [ ]:
xgb_predictor.delete_endpoint(delete_endpoint_config=True)
print("XGBoost endpoint deleted.")

---
# STAGE 2 -- Other Built-in Algorithms

Stage 1 was **supervised** -- we had labels and trained a classifier. But what if you do not have labels? What if you want to discover structure in the data or flag outliers automatically?

SageMaker offers several unsupervised built-in algorithms. We will use two on the fraud data:
- **K-Means Clustering:** Find natural groupings in transaction features
- **Random Cut Forest:** Detect anomalies (statistical outliers) without labels

We also survey two additional built-in algorithms conceptually:
- **BlazingText / Word2Vec:** NLP -- word embeddings and text classification
- **DeepAR:** Time series forecasting with autoregressive RNNs

## K-Means Clustering

K-Means partitions data into **k** clusters by minimizing the distance from each point to its cluster center (centroid). The algorithm does not know about fraud -- it finds groupings based on feature values alone.

**For FraudShield:** Clusters might reveal natural transaction segments -- small retail purchases, large international transfers, suspicious late-night transactions. If one cluster happens to contain mostly fraudulent transactions, we learn something about the structure of fraud without ever providing labels.

**SageMaker's built-in K-Means** is optimized for large datasets, supports GPU acceleration, and uses the same estimator pattern as XGBoost.

### STEP 7 -- Configure and Train K-Means

> **What is happening:** We prepare the fraud features (no target column -- this is unsupervised) and train K-Means with k=4 clusters. The data format for K-Means is different from XGBoost: it expects RecordIO-protobuf or CSV with features only.

In [ ]:
import io
import sagemaker.amazon.common as smac

train_features = train_df[feature_cols].values.astype(np.float32)
val_features = val_df[feature_cols].values.astype(np.float32)
val_labels = val_df["target"].values

buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, train_features)
buf.seek(0)

kmeans_train_key = f"{prefix}/kmeans/train/train.recordio"
s3.upload_fileobj(buf, bucket, kmeans_train_key)

kmeans_train_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/kmeans/train/",
    content_type="application/x-recordio-protobuf",
)

print(f"K-Means training data uploaded: s3://{bucket}/{kmeans_train_key}")
print(f"Feature matrix shape: {train_features.shape}")

In [ ]:
kmeans_image = image_uris.retrieve("kmeans", region)
print(f"K-Means container image: {kmeans_image}")

kmeans_estimator = Estimator(
    image_uri=kmeans_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/kmeans/output",
    sagemaker_session=session,
)

kmeans_estimator.set_hyperparameters(
    k=4,
    feature_dim=len(feature_cols),
    mini_batch_size=200,
)

kmeans_estimator.fit({"train": kmeans_train_input}, logs=True)

print(f"\nK-Means training job: {kmeans_estimator.latest_training_job.name}")

### STEP 8 -- Visualize Cluster Assignments

> **What is happening:** We deploy the K-Means model, predict cluster assignments for the validation data, and visualize them in a 2D scatter plot using the first two features (amount and hour). We overlay the actual fraud labels to see whether clusters capture fraud signal.

In [ ]:
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import CSVSerializer

kmeans_predictor = kmeans_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)

result = kmeans_predictor.predict(val_features)
cluster_labels = np.array([r["closest_cluster"]["predicted_label"] for r in result["predictions"]])

print(f"Cluster distribution: {dict(zip(*np.unique(cluster_labels, return_counts=True)))}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter1 = axes[0].scatter(
    val_features[:, 0], val_features[:, 1],
    c=cluster_labels, cmap="viridis", alpha=0.6, s=20,
)
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Hour")
axes[0].set_title("K-Means Cluster Assignments")
plt.colorbar(scatter1, ax=axes[0], label="Cluster")

colors = ["steelblue" if y == 0 else "crimson" for y in val_labels]
axes[1].scatter(
    val_features[:, 0], val_features[:, 1],
    c=colors, alpha=0.6, s=20,
)
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Hour")
axes[1].set_title("Actual Labels (blue=non-fraud, red=fraud)")

plt.tight_layout()
plt.show()

print("\nFraud rate by cluster:")
for c in sorted(np.unique(cluster_labels)):
    mask = cluster_labels == c
    fraud_rate = val_labels[mask].mean()
    print(f"  Cluster {int(c)}: {mask.sum()} transactions, {fraud_rate:.1%} fraud")

> **Discussion:** Do the clusters separate fraud from non-fraud? If one cluster has a significantly higher fraud rate, what could FraudShield do with that information even without a supervised model? (Segment-level rules, alert thresholds, targeted review queues, prioritizing human review for high-fraud-rate segments.)

In [ ]:
kmeans_predictor.delete_endpoint(delete_endpoint_config=True)
print("K-Means endpoint deleted.")

## Random Cut Forest for Anomaly Detection

Random Cut Forest (RCF) is SageMaker's built-in **anomaly detection** algorithm. It assigns an anomaly score to each data point based on how "easy" it is to isolate from the rest of the data. Points that are far from the normal distribution -- outliers -- get high scores.

**For FraudShield:** High-anomaly transactions are candidates for fraud review. Unlike K-Means (which groups), RCF specifically identifies outliers. In production, you could combine RCF anomaly scores with XGBoost predictions for a two-layer defense:
1. XGBoost flags likely fraud based on learned patterns
2. RCF flags unusual transactions that may be novel fraud types the classifier has not seen

### STEP 9 -- Configure and Train Random Cut Forest

> **What is happening:** We train RCF on the fraud features (unsupervised -- no target column). RCF builds a forest of random cut trees. Each tree recursively partitions the feature space with random axis-aligned cuts. Points that require fewer cuts to isolate are more anomalous.

In [ ]:
buf_rcf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf_rcf, train_features)
buf_rcf.seek(0)

rcf_train_key = f"{prefix}/rcf/train/train.recordio"
s3.upload_fileobj(buf_rcf, bucket, rcf_train_key)

rcf_train_input = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/rcf/train/",
    content_type="application/x-recordio-protobuf",
)

rcf_image = image_uris.retrieve("randomcutforest", region)
print(f"RCF container image: {rcf_image}")

rcf_estimator = Estimator(
    image_uri=rcf_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/rcf/output",
    sagemaker_session=session,
)

rcf_estimator.set_hyperparameters(
    num_trees=50,
    num_samples_per_tree=256,
    feature_dim=len(feature_cols),
)

rcf_estimator.fit({"train": rcf_train_input}, logs=True)

print(f"\nRCF training job: {rcf_estimator.latest_training_job.name}")

### STEP 10 -- Interpret Anomaly Scores

> **What is happening:** We deploy the RCF model, obtain anomaly scores for the validation data, and compare them to the actual fraud labels. High anomaly scores should correlate with fraudulent transactions.

In [ ]:
rcf_predictor = rcf_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)

rcf_result = rcf_predictor.predict(val_features)
anomaly_scores = np.array([r["score"] for r in rcf_result["predictions"]])

print(f"Anomaly score range: [{anomaly_scores.min():.2f}, {anomaly_scores.max():.2f}]")
print(f"Mean score (non-fraud): {anomaly_scores[val_labels == 0].mean():.2f}")
print(f"Mean score (fraud):     {anomaly_scores[val_labels == 1].mean():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(anomaly_scores[val_labels == 0], bins=30, alpha=0.7, label="Non-Fraud", color="steelblue")
axes[0].hist(anomaly_scores[val_labels == 1], bins=30, alpha=0.7, label="Fraud", color="crimson")
axes[0].set_xlabel("Anomaly Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Anomaly Score Distribution by Label")
axes[0].legend()

scatter = axes[1].scatter(
    val_features[:, 0], val_features[:, 1],
    c=anomaly_scores, cmap="YlOrRd", alpha=0.6, s=20,
)
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Hour")
axes[1].set_title("Anomaly Scores (darker = more anomalous)")
plt.colorbar(scatter, ax=axes[1], label="Anomaly Score")

plt.tight_layout()
plt.show()

threshold = np.percentile(anomaly_scores, 90)
flagged = anomaly_scores >= threshold
print(f"\nThreshold (90th percentile): {threshold:.2f}")
print(f"Flagged as anomalous: {flagged.sum()} / {len(flagged)}")
print(f"Fraud rate among flagged: {val_labels[flagged].mean():.1%}")
print(f"Fraud rate overall:       {val_labels.mean():.1%}")

> **Discussion:** Notice that RCF does not know what fraud is -- it identifies statistical outliers. If the fraud rate among flagged transactions is higher than the overall rate, RCF is capturing fraud signal. In production, how would you combine RCF with XGBoost? (Two-layer defense: XGBoost catches known fraud patterns, RCF catches novel/unseen patterns.)

In [ ]:
rcf_predictor.delete_endpoint(delete_endpoint_config=True)
print("RCF endpoint deleted.")

## BlazingText and Word2Vec -- Conceptual Overview

**BlazingText** is SageMaker's built-in NLP algorithm. It provides two modes:

1. **Word2Vec mode:** Learns dense vector representations (embeddings) of words from their context in a corpus. Similar words end up with similar vectors. This is the same Word2Vec algorithm published by Mikolov et al. (2013), but SageMaker's implementation is optimized for distributed training on large corpora.

2. **Text classification mode:** Supervised text classification using a shallow neural network. Fast to train, works well when you have labeled text data.

**Connection to Wednesday's transformers:** Transformers (BERT, GPT) replaced Word2Vec for most NLP tasks. Word2Vec produces static embeddings -- each word gets one vector regardless of context. Transformers produce contextual embeddings -- the same word gets different vectors depending on surrounding words. BlazingText remains useful for:
- Very large corpora where transformer fine-tuning is too expensive
- High-throughput, low-latency text classification
- Generating embeddings for downstream non-neural models

**For FraudShield:** If transaction descriptions or merchant names were text fields, BlazingText could classify them or generate embeddings that capture semantic similarity between merchants.

## DeepAR -- Conceptual Overview

**DeepAR** is SageMaker's built-in time series forecasting algorithm. It uses autoregressive RNNs to generate **probabilistic forecasts** -- not just a single predicted value, but a distribution (mean, quantiles, confidence intervals).

**Key properties:**
- Trains on **multiple related time series** simultaneously (e.g., daily transaction volumes for 100 merchants)
- Learns patterns shared across all series (seasonality, trends) and series-specific patterns
- Outputs quantile forecasts (P10, P50, P90) for uncertainty estimation

**Connection to Monday's RNNs:** DeepAR uses the same recurrent architecture (LSTM) under the hood. The autoregressive component means the model feeds its own predictions back as inputs for future time steps.

**For FraudShield:** DeepAR could forecast expected daily transaction volumes or fraud rates. Significant deviations between forecasted and actual values signal emerging fraud patterns -- a form of time-series anomaly detection that complements the point-in-time detection from XGBoost and RCF.

> **Discussion:** We now have four approaches to fraud detection in SageMaker: XGBoost (supervised classification), K-Means (clustering), RCF (anomaly detection), and DeepAR (time series). In a production system, how would you combine them? Which would run in real-time vs batch?

---
# STAGE 3 -- Hyperparameter Optimization

In Stage 1 you trained XGBoost with manually chosen hyperparameters: `max_depth=5`, `eta=0.2`, `num_round=100`. Those values were reasonable defaults, but how do we know they are optimal?

**HPO (Hyperparameter Optimization)** automates the search. You define:
1. **Objective metric** -- what to optimize (e.g., `validation:auc`)
2. **Parameter ranges** -- the search space for each hyperparameter
3. **Strategy** -- how to explore the space (Bayesian or random)
4. **Budget** -- how many training jobs to run

SageMaker launches multiple training jobs, each with different hyperparameter values. Bayesian optimization learns from prior trials to focus on promising regions of the search space.

**Connection to Wednesday's experiments:** HPO is systematic, automated experimentation. Each HPO trial is an experiment run with its own hyperparameters and metrics. Wednesday you tracked experiments manually. Today, HPO does it for you.

## HPO Concepts

**Objective metric:** The metric the tuner optimizes. It must match a metric that the algorithm logs during training. For XGBoost with `eval_metric="auc"`, the logged metric name is `validation:auc`.

**Parameter ranges:**
- `ContinuousParameter(min, max)` -- for floats like learning rate (`eta`)
- `IntegerParameter(min, max)` -- for integers like tree depth (`max_depth`)
- `CategoricalParameter([...])` -- for discrete choices

**Strategies:**
- **Bayesian:** Builds a probabilistic model of the objective function. After each trial, it updates the model and picks the next hyperparameters where expected improvement is highest. More sample-efficient -- finds better results with fewer trials.
- **Random:** Samples hyperparameters uniformly at random. No learning between trials. Embarrassingly parallel but less efficient.

**Early stopping:** SageMaker can terminate trials early if they fall behind the best result. Saves compute without sacrificing the best outcome.

**Max jobs and max parallel jobs:** Budget controls. `max_jobs` is the total number of training jobs. `max_parallel_jobs` is how many run concurrently. For Bayesian strategy, sequential trials are more informative (each trial learns from all prior), so `max_parallel_jobs` is typically 2-3.

### STEP 11 -- Configure HyperparameterTuner

> **What is happening:** We create a new XGBoost estimator (same as Stage 1 but without fixing `max_depth`, `eta`, and `num_round` -- the tuner will search over those). We wrap it in a `HyperparameterTuner` with defined parameter ranges and the objective metric.

In [ ]:
from sagemaker.tuner import (
    HyperparameterTuner,
    IntegerParameter,
    ContinuousParameter,
)

xgb_hpo_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/xgboost-hpo/output",
    sagemaker_session=session,
)

xgb_hpo_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    subsample=0.8,
    colsample_bytree=0.8,
)

hyperparameter_ranges = {
    "max_depth": IntegerParameter(3, 10),
    "eta": ContinuousParameter(0.01, 0.3),
    "num_round": IntegerParameter(50, 200),
}

tuner = HyperparameterTuner(
    estimator=xgb_hpo_estimator,
    objective_metric_name="validation:auc",
    objective_type="Maximize",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=6,
    max_parallel_jobs=2,
    strategy="Bayesian",
    early_stopping_type="Auto",
)

print("HyperparameterTuner configured:")
print(f"  Objective:        validation:auc (Maximize)")
print(f"  Strategy:         Bayesian")
print(f"  Max jobs:         6")
print(f"  Max parallel:     2")
print(f"  Early stopping:   Auto")
print(f"  Parameter ranges:")
for name, param_range in hyperparameter_ranges.items():
    print(f"    {name}: {param_range}")

### STEP 12 -- Launch HPO Job

> **What is happening:** We call `.fit()` on the tuner with the same train/validation channels from Stage 1. SageMaker launches up to 6 training jobs (2 at a time), each with different hyperparameter values chosen by the Bayesian optimizer. This takes approximately 10-15 minutes.

In [ ]:
tuner.fit(
    {"train": train_input, "validation": val_input},
    logs=False,
)

print(f"HPO job name: {tuner.latest_tuning_job.name}")
print("HPO job complete.")

### STEP 13 -- Analyze HPO Results

> **What is happening:** We retrieve the HPO analytics -- all trials with their hyperparameters and objective metric values. We identify the best trial and compare it to our Stage 1 baseline.

In [ ]:
tuner_analytics = tuner.analytics()
hpo_results = tuner_analytics.dataframe()

display_cols = [
    "TrainingJobName",
    "FinalObjectiveValue",
    "TrainingJobStatus",
]
hp_cols = [c for c in hpo_results.columns if c not in display_cols
           and c in ["max_depth", "eta", "num_round"]]

results_display = hpo_results[display_cols + hp_cols].sort_values(
    "FinalObjectiveValue", ascending=False
)
print("=== HPO Trial Results (sorted by AUC) ===")
print(results_display.to_string(index=False))

In [ ]:
best_job = hpo_results.loc[hpo_results["FinalObjectiveValue"].idxmax()]

print("=== Best HPO Trial ===")
print(f"  Training job: {best_job['TrainingJobName']}")
print(f"  AUC:          {best_job['FinalObjectiveValue']:.4f}")
if "max_depth" in best_job:
    print(f"  max_depth:    {best_job['max_depth']}")
if "eta" in best_job:
    print(f"  eta:          {best_job['eta']}")
if "num_round" in best_job:
    print(f"  num_round:    {best_job['num_round']}")

print("\n=== Stage 1 Baseline ===")
print(f"  max_depth: 5, eta: 0.2, num_round: 100")
print(f"  (Compare the AUC above to Stage 1's AUC to see the HPO improvement.)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, param in enumerate(["max_depth", "eta", "num_round"]):
    if param in hpo_results.columns:
        axes[idx].scatter(
            hpo_results[param].astype(float),
            hpo_results["FinalObjectiveValue"],
            c="steelblue", s=60, edgecolors="black", linewidth=0.5,
        )
        axes[idx].set_xlabel(param)
        axes[idx].set_ylabel("AUC")
        axes[idx].set_title(f"AUC vs {param}")

plt.tight_layout()
plt.show()

## Connecting HPO to Wednesday's Experiment Tracking

On Wednesday you learned to track experiments with SageMaker Experiments -- manually logging hyperparameters, metrics, and artifacts for each training run. HPO automates that process:

| Aspect | Manual Experiments (Wednesday) | HPO (Today) |
|--------|-------------------------------|-------------|
| **Who chooses hyperparameters** | You | Bayesian optimizer |
| **Number of trials** | As many as you run | Configured (`max_jobs`) |
| **Learning between trials** | Your intuition | Probabilistic model |
| **Tracking** | Explicit logging | Automatic |
| **Best practice** | Exploration, understanding | Exploitation, optimization |

**Production pattern:** Use manual experiments to explore the problem space and narrow down reasonable hyperparameter ranges. Then use HPO to fine-tune within those ranges. Log both manual experiments and HPO trials in the same experiment for a complete record.

> **Discussion:** Wednesday you manually tracked experiments. Today HPO automated the process. What are the trade-offs? (Manual gives intuition and control -- you learn what matters. HPO is systematic but costs more compute. Best practice: manual exploration first, then HPO to fine-tune.)

## Cleanup

> **What is happening:** We clean up any resources created during this notebook. HPO may have created model artifacts but we did not deploy any endpoints in Stage 3. We verify no active endpoints remain from any stage.

In [ ]:
sm_client = boto3.client("sagemaker")

endpoints = sm_client.list_endpoints(
    StatusEquals="InService",
    MaxResults=20,
)["Endpoints"]

fraudshield_endpoints = [
    ep for ep in endpoints
    if "xgboost" in ep["EndpointName"].lower()
    or "kmeans" in ep["EndpointName"].lower()
    or "randomcutforest" in ep["EndpointName"].lower()
    or "rcf" in ep["EndpointName"].lower()
]

if fraudshield_endpoints:
    print("Deleting active endpoints:")
    for ep in fraudshield_endpoints:
        name = ep["EndpointName"]
        print(f"  Deleting {name}...")
        sm_client.delete_endpoint(EndpointName=name)
        try:
            sm_client.delete_endpoint_config(EndpointConfigName=name)
        except Exception:
            pass
    print("All endpoints deleted.")
else:
    print("No active FraudShield endpoints found. All clean.")

print("\nVerify in the SageMaker console: Inference > Endpoints -- no active endpoints.")

---
## Wrap-up and Friday Preview

**Today you accomplished three things:**

1. **Trained XGBoost** as a built-in algorithm on the same fraud data from Monday. No training script -- just data and hyperparameters. You compared the results to Monday's Random Forest and saw how gradient boosting's sequential correction mechanism can outperform independent tree ensembles.

2. **Explored unsupervised algorithms.** K-Means revealed natural clusters in the fraud data. Random Cut Forest flagged anomalies without any labels. You also surveyed BlazingText for NLP and DeepAR for time series -- expanding the algorithm toolkit beyond supervised classification.

3. **Launched HPO** to systematically search for the best XGBoost hyperparameters. Bayesian optimization automated the experimentation process from Wednesday, finding better configurations than manual tuning.

**Friday Preview:** Friday wraps up the week with SageMaker Pipelines and end-to-end ML workflows. You will connect everything -- data preparation, training, evaluation, registration, and deployment -- into an automated pipeline. The manual steps you have been doing all week become reproducible, auditable DAG steps.